In [ ]:
from common import *

## 13. Produkcija modela

Primecujemo da je najbolji model...

In [ ]:
import os
from mlflow import MlflowClient

mlflow.set_tracking_uri(os.environ.get("MLFLOW_TRACKING_URI", "http://127.0.0.1:5000"))
client = MlflowClient(tracking_uri=mlflow.get_tracking_uri())

REGISTROVANI_MODEL_NAME = "WeatherAusRainModel"

# "xgboost_tezinsko_f1" (sveska 08) je preporuceni finalni model - scale_pos_weight +
# F1-optimalan prag, treniran na 10 atributa (Sused_RainToday_pct/Sused_Pressure3pm_avg
# su namerno izostavljeni - najmanje bitna 2 od originalnih 12 atributa, ~2-3% odluke
# modela svaki, a jedina bi u produkciji zahtevala live podatke sa SVIH susednih
# stanica istovremeno). Isti feature set se koristi svuda - u analizi i u produkciji
# (jedini nacin servisiranja je uzivo, videti app/api).
CILJANI_RUN_NAME = "unified_noleak_xgboost_tezinsko_f1"

experiment = client.get_experiment_by_name("WeatherAus proba 1")
kandidati = client.search_runs(
    experiment_ids=[experiment.experiment_id],
    filter_string=f"tags.mlflow.runName = '{CILJANI_RUN_NAME}'",
    order_by=["start_time DESC"],
    max_results=1,
)
if not kandidati:
    raise RuntimeError(
        f"Nije pronadjen nijedan MLflow run sa imenom '{CILJANI_RUN_NAME}' u eksperimentu "
        f"'WeatherAus proba 1' - da li je sveska 08_finalne_predikcije uspesno izvrsena?"
    )

# search_runs ne popunjava uvek .outputs (poznato ponasanje u ovoj verziji MLflow-a),
# zato ponovo ucitavamo run preko get_run kako bismo pouzdano dobili model_id.
pobednicki_run = client.get_run(kandidati[0].info.run_id)
model_id = pobednicki_run.outputs.model_outputs[0].model_id
optimalni_prag = float(pobednicki_run.data.params["optimalni_prag"])

registrovan = mlflow.register_model(f"models:/{model_id}", REGISTROVANI_MODEL_NAME)
client.set_registered_model_alias(REGISTROVANI_MODEL_NAME, "champion", registrovan.version)
client.set_model_version_tag(REGISTROVANI_MODEL_NAME, registrovan.version, "optimalni_prag", str(optimalni_prag))

print(f"Registrovan '{REGISTROVANI_MODEL_NAME}' v{registrovan.version} (run {pobednicki_run.info.run_id}) sa aliasom 'champion'")
print(f"Optimalni prag odlucivanja koji API koristi: {optimalni_prag}")


In [ ]:
# Poslednji korak pipeline-a: obavesti FastAPI servis (app/api) da je nova champion
# verzija spremna, tako da je ucita bez restarta kontejnera. API ne mora da bude gore
# da bi ovaj task uspeo (npr. prvi ikad pokretani DAG run, pre nego sto je iko podigao
# docker-compose api servis) - to je "nice to have" korak, ne kriticna zavisnost
# pipeline-a, pa neuspeh ovde samo upozorava, ne pada ceo task.
import requests

API_RELOAD_URL = os.environ.get("API_RELOAD_URL", "http://api:8000/reload")
try:
    odgovor = requests.post(API_RELOAD_URL, timeout=15)
    odgovor.raise_for_status()
    print(f"API osvezen: {odgovor.json()}")
except Exception as e:
    print(f"[upozorenje] Nisam mogao da osvezim API na {API_RELOAD_URL} ({e}). "
          f"Ako je 'api' servis gore, osvezi rucno: curl -X POST {API_RELOAD_URL}")
